## Week 4 Notebook 1: Create synthetic questions as ground truth data
* Create synthetic questions as ground truth data
* Use structured outputs with `openai.responses.parse()`
* Set up retry loops (max retries = 3) for if LLM requests fails because of temp API or network issue `llm_structure_retry`

In [1]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main"

! wget $PREFIX/01-agentic-rag/code/ingest.py
! wget $PREFIX/01-agentic-rag/code/rag_helper.py
! wget $PREFIX/04-evaluation/code/evaluation_utils.py

--2026-07-06 11:24:15--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py.1’

ingest.py.1         100%[===================>]     738  --.-KB/s    in 0s      

2026-07-06 11:24:16 (46.4 MB/s) - ‘ingest.py.1’ saved [738/738]

--2026-07-06 11:24:16--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response..

In [1]:
from ingest import load_faq_data

documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

print(len(documents_llm))

103


In [3]:
documents = documents_llm
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [4]:
# use structured outputs so that it's easier to parse outputs from llms
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
import json

user_prompt = json.dumps(doc)

In [8]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt},
]

In [9]:
# using responses.parse (instead of responses.create) to parse structured responses
response = openai_client.responses.parse(
    model="gpt-5.4-mini", input=messages, text_format=Questions
)

response.output_parsed.questions

['I found this course late — is it still possible to join now?',
 'Can I still enroll even if the course has already started?',
 'If I join after the course is underway, can I still get a certificate?',
 'Do I need to finish and submit the project while submissions are open to qualify for the certificate?',
 'What happens if I join late — am I still eligible for course completion credit or certification?']

## Reusable utilities

We'll need this pattern again in other evaluation sections today, so
we put it in a reusable helper.

It contains helper functions we'll reuse in this module:

- `llm_structured`: calls the OpenAI API with structured output
- `llm_structured_retry`: retries structured-output calls when a
  request fails
- `calc_price`: calculates the price from token usage
- `calc_total_price`: calculates the total price from multiple usage
  objects
- `map_progress`: runs work in parallel and tracks progress. We'll use it
  in the next lesson.

In [10]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client, data_gen_instructions, user_prompt, output_type=Questions
)

print(result.questions)

['I just found this course late — can I still sign up and follow along?', 'Is it too late to join if the course has already started?', 'Can I still take part now, and what do I need to do if I want the certificate?', 'If I join after the start date, am I still eligible for a certificate?', 'What’s the deadline for project submission if I want to get the certificate?']


In [11]:
# track tokens used
usage.input_tokens, usage.output_tokens

(207, 96)

In [12]:
# track tokens & costs
from evaluation_utils import calc_price

cost = calc_price(usage)
cost

{'input_cost': 0.00015525, 'output_cost': 0.000432, 'total_cost': 0.00058725}

### Convert these new synthetic questions into ground truth records:

In [13]:
records = []

for q in result.questions:
    records.append({"question": q, "document": doc["id"]})

records

[{'question': 'I just found this course late — can I still sign up and follow along?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to join if the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take part now, and what do I need to do if I want the certificate?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the start date, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for project submission if I want to get the certificate?',
  'document': '74eb249bbf'}]

For each document, we:

* convert the document to JSON so we can send it to the LLM
* ask the LLM to return a Questions object
* create one ground truth record for each generated question

Each record contains the generated question and the ID of the document that should answer the question.

When we send many requests, one of them might fail. We don't want the entire batch to fail because of one temporary error.

Import the retry helper from evaluation_utils.py:
```
from evaluation_utils import llm_structured_retry
```

`llm_structured` makes one structured-output call. `llm_structured_retry` wraps the same call in a retry loop. If one request fails because of a temporary API or network issue, it waits briefly and tries again.

In [14]:
from evaluation_utils import llm_structured_retry


def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client, data_gen_instructions, user_prompt, Questions
    )

    results = []

    for q in out.questions:
        results.append({"question": q, "document": doc["id"]})

    return results, usage

In [ ]:
# try creating synthetic ground truth for first 5 docs
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [16]:
# visualise the synthetic questions' structures
import pandas as pd

pd.DataFrame(records)

,question,document
0,How do I find my own entry on the leaderboard ...,c2903069a0
1,What name do I get on the leaderboard when I f...,c2903069a0
2,Where can I check what my display name is for ...,c2903069a0
3,Can I change the name that shows on the leader...,c2903069a0
4,What do I need to fill in if I want my real na...,c2903069a0


## Parallel processing
* We process the documents in parallel and track progress while the requests run.

* One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.


In [22]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

print(len(ground_truth))

515


In [26]:
results[1]

([{'question': 'I signed up for the LLM Zoomcamp, but I never got any confirmation email — is that a problem, or am I already in?',
   'document': '977bf7786c'},
  {'question': 'Do I actually need to wait for an acceptance or confirmation message before I can start the course and submit homework?',
   'document': '977bf7786c'},
  {'question': 'If I filled out the registration form, does that mean I’m officially registered somewhere, or is it only for showing interest?',
   'document': '977bf7786c'},
  {'question': 'Can I begin the LLM Zoomcamp and turn in assignments even if I didn’t register first, as long as the homework form is still open?',
   'document': '977bf7786c'},
  {'question': 'Is the course checking who submitted the registration form against some list, or can I just join and start learning right away?',
   'document': '977bf7786c'}],
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=146, output_tokens_details=OutputTo

with 5 questions genearated per doc, we should get ~5x the num of documents

In [27]:
# calculate the total cost

from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost += cost["total_cost"]

print(total_cost)

0.07739175000000001
